# GapHunter — Chronos Prediction Server

**실행 환경**: Google Colab (Free GPU — T4)

**모델**: Amazon Chronos (TimesFM과 동급 Foundation 시계열 모델)
**변경 이유**: TimesFM이 Colab Python 3.11과 호환성 이슈로 설치 실패. Chronos는 클린 설치됨.

## 실행 순서
1. Runtime → Change runtime type → **T4 GPU**
2. 셀을 위에서 아래로 순서대로 실행
3. Cell 4에서 출력되는 ngrok URL 복사
4. 로컬 `.env`에 `COLAB_PREDICTOR_URL=https://xxxx.ngrok-free.app` 추가

In [ ]:
# Cell 1 — Install dependencies
!pip install chronos-forecasting flask flask-cors pyngrok --quiet

In [ ]:
# Cell 2 — Load Chronos model (T5-small, ~60MB, fastest)
import torch
from chronos import ChronosPipeline

print('Loading Chronos model...')
pipeline = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-small',
    device_map='cuda' if torch.cuda.is_available() else 'cpu',
    torch_dtype=torch.bfloat16,
)
print('Chronos model loaded.')

In [ ]:
# Cell 3 — Define Flask prediction server
import numpy as np
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)


@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': 'chronos-t5-small'})


@app.route('/predict', methods=['POST'])
def predict():
    try:
        data    = request.get_json(force=True)
        series  = data.get('time_series', [])
        horizon = int(data.get('horizon', 30))

        if not series:
            return jsonify({'error': 'time_series is required'}), 400

        context = torch.tensor(series, dtype=torch.float32)

        # Chronos returns shape (num_samples, horizon)
        forecast = pipeline.predict(
            context=context,
            prediction_length=horizon,
            num_samples=20,
        )

        # forecast shape: (1, num_samples, horizon)
        samples = forecast[0].cpu().numpy()   # (num_samples, horizon)

        median = np.median(samples, axis=0).tolist()
        lower  = np.quantile(samples, 0.1, axis=0).tolist()
        upper  = np.quantile(samples, 0.9, axis=0).tolist()

        return jsonify({
            'forecast':         median,
            'confidence_lower': lower,
            'confidence_upper': upper,
            'model':            'chronos',
        })

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500


print('Flask app defined. Proceed to Cell 4.')

In [ ]:
# Cell 4 — Start server + expose via ngrok
# 1. https://dashboard.ngrok.com 에서 Authtoken 복사
# 2. 아래 NGROK_AUTH_TOKEN에 붙여넣기

from pyngrok import ngrok
import threading

NGROK_AUTH_TOKEN = ''  # <-- 여기에 ngrok 토큰 붙여넣기

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

def run_flask():
    app.run(host='0.0.0.0', port=5000, use_reloader=False)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()

public_url = ngrok.connect(5000)
print('='*60)
print('Chronos server running!')
print(f'Public URL: {public_url}')
print()
print('로컬 .env에 아래를 추가하세요:')
print(f'COLAB_PREDICTOR_URL={public_url}')
print('='*60)

In [ ]:
# Cell 5 — 서버 동작 확인
import requests

resp = requests.get('http://localhost:5000/health')
print('Health:', resp.json())

test_series = [45, 50, 48, 55, 60, 58, 62, 70, 68, 75, 72, 80]
resp = requests.post('http://localhost:5000/predict', json={
    'time_series': test_series,
    'horizon': 7
})
result = resp.json()
print('Forecast (7 days):', [round(v, 1) for v in result['forecast']])
print('Lower:            ', [round(v, 1) for v in result['confidence_lower']])
print('Upper:            ', [round(v, 1) for v in result['confidence_upper']])
print('Model:            ', result['model'])